In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

# ==========================================
# 1. DATA LOADING & SETUP
# ==========================================
DATA_DIR = "/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge"
train_images_path = os.path.join(DATA_DIR, "train_images")
test_images_path = os.path.join(DATA_DIR, "test_images")
train_labels = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))

with open(os.path.join(DATA_DIR, "classes.txt")) as f:
    classes = [line.strip() for line in f]

class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

# ==========================================
# 2. TRANSFORMATIONS
# ==========================================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(), 
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

val_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

# ==========================================
# 3. DATASETS
# ==========================================
class ShiftGuardDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, train=True):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.train = train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id']).zfill(6)
        img_path = os.path.join(self.img_dir, img_id + ".png")
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        if self.train:
            label = class_to_idx[self.df.iloc[idx]['label']]
            return image, label
        else:
            return image, img_id

class TestDataset(Dataset):
    def __init__(self, img_dir, transform):
        self.img_dir = img_dir
        self.transform = transform
        self.ids = sorted(os.listdir(img_dir))
        
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        img_name = self.ids[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        img_id = img_name.replace(".png", "")
        return image, img_id

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        
        self.se = SEBlock(out_c)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return torch.relu(out)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_c = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        self.layer1 = self._make_layer(96, 3, 1)
        self.layer2 = self._make_layer(192, 3, 2)
        self.layer3 = self._make_layer(384, 3, 2)
        self.layer4 = self._make_layer(768, 3, 2)
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(768, 10)
        
    def _make_layer(self, out_c, blocks, stride):
        layers = []
        layers.append(BasicBlock(self.in_c, out_c, stride))
        self.in_c = out_c
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_c, out_c))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ==========================================
# 5. EVALUATION FUNCTION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            pred = torch.argmax(outputs, 1)
            preds.extend(pred.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return f1_score(targets, preds, average='macro')

# ==========================================
# 6. K-FOLD CROSS VALIDATION WITH SWA & CUTMIX
# ==========================================
k_folds = 5  
epochs = 120 
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_model_paths = []


# V2 Transform for Mixup and CutMix
cutmix_or_mixup = v2.RandomChoice([
    v2.CutMix(num_classes=10, alpha=1.0),
    v2.MixUp(num_classes=10, alpha=1.0) 
])

for fold, (train_idx, val_idx) in enumerate(skf.split(train_labels, train_labels['label'])):
    print(f"\n{'='*20} FOLD {fold+1}/{k_folds} {'='*20}")
    
    train_fold_df = train_labels.iloc[train_idx].reset_index(drop=True)
    val_fold_df = train_labels.iloc[val_idx].reset_index(drop=True)
    
    train_dataset = ShiftGuardDataset(train_fold_df, train_images_path, train_transform, True)
    val_dataset   = ShiftGuardDataset(val_fold_df, train_images_path, val_transform, True)
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    
    model = CNN().to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=0.1,  
        steps_per_epoch=len(train_loader), 
        epochs=epochs
    )
    
    # --- SETUP SWA ---
    swa_model = AveragedModel(model)
    swa_start = int(epochs * 0.75) # Start averaging in the last 25% of epochs
    swa_scheduler = SWALR(optimizer, swa_lr=0.01)
    
   
    scaler = torch.amp.GradScaler('cuda')
    model_save_path = f"swa_model_fold_{fold+1}.pth"
    fold_model_paths.append(model_save_path)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Apply CutMix or Mixup to the batch
            images, labels = cutmix_or_mixup(images, labels)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            
            # Update correct scheduler depending on the epoch
            if epoch < swa_start:
                scheduler.step()
                
        # SWA Parameter Update
        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
            
        # Optional: Print standard model validation every 10 epochs to track progress
        if (epoch + 1) % 10 == 0:
            val_f1 = evaluate(model, val_loader)
            print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f} - Std Model F1: {val_f1:.4f}")

    print(f"Updating Batch Norm statistics for SWA Model on Fold {fold+1}...")
    update_bn(train_loader, swa_model, device=device)
    
    # Evaluate SWA model
    swa_f1 = evaluate(swa_model, val_loader)
    print(f"Fold {fold+1} Final SWA F1: {swa_f1:.4f} -> Model saved.")
    
    # Save the averaged SWA model
    torch.save(swa_model.state_dict(), model_save_path)

# ==========================================
# 7. ENSEMBLED INFERENCE WITH SWA & TTA
# ==========================================
print("\nStarting Ensembled Inference with Test-Time Augmentation (TTA)...")

test_dataset = TestDataset(test_images_path, val_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

ensemble_models = []
for model_path in fold_model_paths:
    # Must wrap your CNN in AveragedModel to properly load the SWA state_dict
    m = AveragedModel(CNN()).to(device)
    m.load_state_dict(torch.load(model_path))
    m.eval()
    ensemble_models.append(m)

predictions = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        batch_probs = torch.zeros((images.size(0), 10)).to(device)
        
        for model in ensemble_models:
            with torch.amp.autocast('cuda'):
                # TTA 1: Original Image
                outputs_orig = model(images)
                batch_probs += F.softmax(outputs_orig, dim=1)
                
                # TTA 2: Horizontally Flipped Image
                images_flipped = torch.flip(images, dims=[3])
                outputs_flipped = model(images_flipped)
                batch_probs += F.softmax(outputs_flipped, dim=1)
            
        batch_probs /= (k_folds * 2)
        preds = torch.argmax(batch_probs, 1).cpu().numpy()
        
        for i, p in enumerate(preds):
            predictions.append((ids[i], idx_to_class[p]))

# ==========================================
# 8. SUBMISSION
# ==========================================
submission = pd.DataFrame(predictions, columns=["id", "label"])
submission.to_csv("submission.csv", index=False)
print("\nEnsemble submission saved successfully!")
print(submission.head())


==================== FOLD 1/5 ====================
Epoch 10/120 - Loss: 1.5659 - Std Model F1: 0.4414
Epoch 20/120 - Loss: 1.4969 - Std Model F1: 0.4545
Epoch 30/120 - Loss: 1.4335 - Std Model F1: 0.5137
Epoch 40/120 - Loss: 1.4007 - Std Model F1: 0.5596
Epoch 50/120 - Loss: 1.4260 - Std Model F1: 0.5610
Epoch 60/120 - Loss: 1.3883 - Std Model F1: 0.6169
Epoch 70/120 - Loss: 1.3710 - Std Model F1: 0.6345
Epoch 80/120 - Loss: 1.3310 - Std Model F1: 0.5755
Epoch 90/120 - Loss: 1.2924 - Std Model F1: 0.7468
Epoch 100/120 - Loss: 1.1939 - Std Model F1: 0.7947
Epoch 110/120 - Loss: 1.1406 - Std Model F1: 0.8241
Epoch 120/120 - Loss: 1.1345 - Std Model F1: 0.8272
Updating Batch Norm statistics for SWA Model on Fold 1...
Fold 1 Final SWA F1: 0.8512 -> Model saved.

==================== FOLD 2/5 ====================
Epoch 10/120 - Loss: 1.5530 - Std Model F1: 0.4658
Epoch 20/120 - Loss: 1.4894 - Std Model F1: 0.4500
Epoch 30/120 - Loss: 1.4695 - Std Model F1: 0.5745
Epoch 40/120 - Loss: 1.440